# Import packages

In [1]:
import os
import pandas as pd
import requests
import base64
import time
import statistics

# Global Variable

In [2]:
# Silakan diubah
CONTAINER_NAME = "No-Container" # 
TASK_NAME = "segmentation" # classification | segementation
ALGORITHM_NAME = "upernet-swin-tiny" # 
API_URL = "http://192.168.1.55:5000/segment" # /classify | /segment

# Jangan diubah
DATATEST_PATH = f"./datatest/segmentation/"
column_names = ['iterasi', 'cpu_usage', 'memory_usage', 'processing_time', 'latency']
df_result = pd.DataFrame(columns=column_names)
df_result

,iterasi,cpu_usage,memory_usage,processing_time,latency


# Load data gambar (500 gambar/tugas)

In [3]:
# images = []

# for i in range(1, 501):
#     # Load image from datatest folder
#     image_path = f"{DATATEST_PATH}{i}.jpg"
#     with open(image_path, "rb") as image_file: 
#         # Convert images to base64
#         image_data = image_file.read()
        
#         # Append image to images list
#         images.append(image_data)

images = []
for filename in os.listdir(DATATEST_PATH):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        with open(os.path.join(DATATEST_PATH, filename), "rb") as f:
            image_base64 = base64.b64encode(f.read()).decode("utf-8")
            images.append(image_base64)

df_result = pd.DataFrame()


# Hit API Endpoint untuk setiap gambar dan masukkan ke dalam DataFrame

In [4]:
# for iter, image in enumerate(images): 
#     response = requests.post(API_URL, data={
#         "image_data" : image,
#         "model_name": ALGORITHM_NAME
#     })
#     result = response.json()
#     data = response['stats']
#     latency = response.elapsed.total_seconds() * 1000  # Convert to milliseconds
#     data['latency'] = latency
#     df_result = pd.concat([df_result, pd.DataFrame([data])], ignore_index=True)

for idx, image in enumerate(images, start=1):
    try:
        response = requests.post(API_URL, json={
            "image_data": image,
            "model_name": ALGORITHM_NAME
        })

        if response.status_code == 200:
            result = response.json()
            data = result["stats"]
            data["latency"] = response.elapsed.total_seconds() * 1000
            df_result = pd.concat([df_result, pd.DataFrame([data])], ignore_index=True)
        else:
            print(f"Error pada gambar ke-{idx}: {response.status_code}")

    except Exception as e:
        print(f"Exception pada gambar ke-{idx}: {e}")

# Save DataFrame ke local

In [5]:
df_result.to_csv(f"{CONTAINER_NAME}_{TASK_NAME}_{ALGORITHM_NAME}.csv", index=False)